<a href="https://colab.research.google.com/github/DanL1L/Scrape/blob/main/0_nowcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import os
import pandas as pd
import json
import glob
import re
import numpy as np

os.chdir(r'/content/sample_data')

In [18]:
# import map
map = pd.read_excel(r'/content/sample_data/map.xlsx')
map['product_sku'] = map['product_sku'].astype(str)
map['code'] = map['code'].apply(lambda x: x[:3] if pd.notna(x) and len(x) > 3 else x)    # categories on the same level F011 -> F01
map = map[map['confidence'] >= 90]
map

,product_sku,product_name,code,category_name,confidence
0,194813,BOSTAVAN Dry white wine DOR Feteasca Alba & Ch...,F15,....wine,100
1,2560,CRICOVA Dry red wine PRESTIGE Cabernet Sauvign...,F15,....wine,100
2,278829,RADACINI Sparkling Wine white semidry750 ml,F15,....wine,100
3,3238,CALARASI Strong alcoholic drink DUMBRAVA 500ml,F15,..Alcoholic drinks,95
4,13256,CALARASI Divin LEGENDA 3 years 500ml,F15,..Alcoholic drinks,95
...,...,...,...,...,...
9643,239078,KAMIS Ground Nutmeg 15g,F16,..Other foodstuffs not classified in other cat...,95
9644,239016,KAMIS Cinnamon Sticks 17g,F16,..Other foodstuffs not classified in other cat...,95
9645,488467,KAMIS Vanilla pod 1 pc,F16,..Other foodstuffs not classified in other cat...,95
9646,2010868,KAMIS Ground cardamom 10g,F16,..Other foodstuffs not classified in other cat...,95


In [20]:
# import weights
weights = pd.read_excel(r'/content/sample_data/weight_data.xlsx', sheet_name = 'w_m')
weights['date'] = pd.to_datetime(weights['date'])
weights = weights.melt(id_vars=['date'], var_name='code', value_name='weight')   # transform data from wide to long
weights = weights.rename(columns={"date": "month"})
weights

,month,code,weight
0,2015-01-01,T,1.000000
1,2015-02-01,T,1.000000
2,2015-03-01,T,1.000000
3,2015-04-01,T,1.000000
4,2015-05-01,T,1.000000
...,...,...,...
14659,2027-08-01,S12,0.000641
14660,2027-09-01,S12,0.000641
14661,2027-10-01,S12,0.000641
14662,2027-11-01,S12,0.000641


In [23]:
# find all .json files in folder, glob.glob() finds all data paths
files = glob.glob(os.path.join(r'/content/sample_data/Linella', '*.json'))

In [24]:
scrape_data = []

for file in files:
    # date from name e.g. linella_scrape_20260402.json
    match = re.search(r'\d{8}', os.path.basename(file))
    date = match.group(0)

    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

        for item in data:
            # if there is discount price take discount price, otherwise take product_price
            d_price = item.get('discount_price')
            p_price = item.get('product_price')
            final_price = d_price if d_price else p_price

            scrape_data.append({
                'date': date,
                'product_sku': item.get('product_sku'),
                'price': final_price
            })

In [ ]:
scrape_data

[{'date': '20260402', 'product_sku': '2737', 'price': '149.00'},
 {'date': '20260402', 'product_sku': '2570', 'price': '175.00'},
 {'date': '20260402', 'product_sku': '15438', 'price': '509.00'},
 {'date': '20260402', 'product_sku': None, 'price': None},
 {'date': '20260402', 'product_sku': '187848', 'price': '309.00'},
 {'date': '20260402', 'product_sku': '40229', 'price': '79.90'},
 {'date': '20260402', 'product_sku': '2680', 'price': '149.00'},
 {'date': '20260402', 'product_sku': None, 'price': None},
 {'date': '20260402', 'product_sku': '194813', 'price': '95.30'},
 {'date': '20260402', 'product_sku': '2511', 'price': '245.00'},
 {'date': '20260402', 'product_sku': '2513', 'price': '275.00'},
 {'date': '20260402', 'product_sku': '120168', 'price': '279.00'},
 {'date': '20260402', 'product_sku': '2908', 'price': '235.00'},
 {'date': '20260402', 'product_sku': '2907', 'price': '175.00'},
 {'date': '20260402', 'product_sku': None, 'price': None},
 {'date': '20260402', 'product_sku': 

In [25]:
# from list to dataframe
scrape_data_df = pd.DataFrame(scrape_data)
scrape_data_df['price'] = pd.to_numeric(scrape_data_df['price'], errors='coerce')
scrape_data_df['date'] = pd.to_datetime(scrape_data_df['date'], format='%Y%m%d')
scrape_data_df['product_sku'] = scrape_data_df['product_sku'].astype(str)
scrape_data_df

,date,product_sku,price
0,2026-04-07,15438,509.0
1,2026-04-07,None,NaN
2,2026-04-07,2680,175.0
3,2026-04-07,128213,89.9
4,2026-04-07,None,NaN
...,...,...,...
569599,2026-05-29,None,NaN
569600,2026-05-29,None,NaN
569601,2026-05-29,None,NaN
569602,2026-05-29,None,NaN


In [26]:
scrape_map = pd.merge(scrape_data_df, map[['code', 'product_sku']], how='left', on='product_sku')
scrape_map.dropna(subset=['code'], inplace=True)
scrape_map

,date,product_sku,price,code
0,2026-04-07,15438,509.0,F15
3,2026-04-07,128213,89.9,F15
5,2026-04-07,58038,206.0,F15
14,2026-04-07,74310,73.9,F15
16,2026-04-07,2908,235.0,F15
...,...,...,...,...
569330,2026-05-29,488467,199.0,F16
569331,2026-05-29,239022,74.9,F16
569333,2026-05-29,239016,49.9,F16
569337,2026-05-29,416666,37.0,N13


In [27]:
# add month column
scrape_map['month'] = scrape_map['date'].dt.to_period('M').dt.to_timestamp()
scrape_map

,date,product_sku,price,code,month
0,2026-04-07,15438,509.0,F15,2026-04-01
3,2026-04-07,128213,89.9,F15,2026-04-01
5,2026-04-07,58038,206.0,F15,2026-04-01
14,2026-04-07,74310,73.9,F15,2026-04-01
16,2026-04-07,2908,235.0,F15,2026-04-01
...,...,...,...,...,...
569330,2026-05-29,488467,199.0,F16,2026-05-01
569331,2026-05-29,239022,74.9,F16,2026-05-01
569333,2026-05-29,239016,49.9,F16,2026-05-01
569337,2026-05-29,416666,37.0,N13,2026-05-01


In [28]:
# aggregate on month - product_sku level (code added in groupby so it doesnt disappear but it changes nothing)
scrape_map_agg = scrape_map.groupby(['month', 'product_sku', 'code'], as_index=False).agg({
    'price': 'mean'
})
scrape_map_agg = scrape_map_agg.sort_values(by=['product_sku', 'month'])
scrape_map_agg

,month,product_sku,code,price
0,2026-04-01,10019,N15,49.400000
7954,2026-05-01,10019,N15,46.650000
15543,2026-06-01,10019,N15,49.400000
22998,2026-07-01,10019,N15,49.400000
1,2026-04-01,10029,N15,54.950000
...,...,...,...,...
30267,2026-07-01,9836,F16,12.900000
7953,2026-04-01,9972,N15,47.750000
15542,2026-05-01,9972,N15,35.754545
22997,2026-06-01,9972,N15,47.811538


In [29]:
# adding prices from previous month
scrape_map_agg['price_prev'] = scrape_map_agg.groupby('product_sku')['price'].shift(1)
scrape_map_agg['month_prev'] = scrape_map_agg.groupby('product_sku')['month'].shift(1)

# if month and month_prev +1 don't match, than previous price is none
is_exact_previous_month = (scrape_map_agg['month_prev'] + pd.DateOffset(months=1)) == scrape_map_agg['month']

scrape_map_agg.loc[~is_exact_previous_month, 'price_prev'] = None
scrape_map_agg

,month,product_sku,code,price,price_prev,month_prev
0,2026-04-01,10019,N15,49.400000,NaN,NaT
7954,2026-05-01,10019,N15,46.650000,49.400000,2026-04-01
15543,2026-06-01,10019,N15,49.400000,46.650000,2026-05-01
22998,2026-07-01,10019,N15,49.400000,49.400000,2026-06-01
1,2026-04-01,10029,N15,54.950000,NaN,NaT
...,...,...,...,...,...,...
30267,2026-07-01,9836,F16,12.900000,12.900000,2026-06-01
7953,2026-04-01,9972,N15,47.750000,NaN,NaT
15542,2026-05-01,9972,N15,35.754545,47.750000,2026-04-01
22997,2026-06-01,9972,N15,47.811538,35.754545,2026-05-01


In [30]:
# price index = price / price from prevoius month
scrape_map_agg['price_index'] = scrape_map_agg['price'] / scrape_map_agg['price_prev']
scrape_map_agg = scrape_map_agg.drop(columns=['month_prev'])
scrape_map_agg.dropna(subset=['price_index'], inplace=True)
scrape_map_agg

,month,product_sku,code,price,price_prev,price_index
7954,2026-05-01,10019,N15,46.650000,49.400000,0.944332
15543,2026-06-01,10019,N15,49.400000,46.650000,1.058950
22998,2026-07-01,10019,N15,49.400000,49.400000,1.000000
7955,2026-05-01,10029,N15,54.950000,54.950000,1.000000
15544,2026-06-01,10029,N15,54.950000,54.950000,1.000000
...,...,...,...,...,...,...
22996,2026-06-01,9836,F16,12.900000,12.900000,1.000000
30267,2026-07-01,9836,F16,12.900000,12.900000,1.000000
15542,2026-05-01,9972,N15,35.754545,47.750000,0.748786
22997,2026-06-01,9972,N15,47.811538,35.754545,1.337216


In [31]:
# merge scrape data with weights and add final code for MoM nowcast
scrape_map_agg_w = pd.merge(scrape_map_agg, weights, how='left', on=['code','month'])
scrape_map_agg_w['final_code'] = scrape_map_agg_w['code'].str[0]
scrape_map_agg_w

,month,product_sku,code,price,price_prev,price_index,weight,final_code
0,2026-05-01,10019,N15,46.650000,49.400000,0.944332,0.006471,N
1,2026-06-01,10019,N15,49.400000,46.650000,1.058950,0.006471,N
2,2026-07-01,10019,N15,49.400000,49.400000,1.000000,0.006471,N
3,2026-05-01,10029,N15,54.950000,54.950000,1.000000,0.006471,N
4,2026-06-01,10029,N15,54.950000,54.950000,1.000000,0.006471,N
...,...,...,...,...,...,...,...,...
22293,2026-06-01,9836,F16,12.900000,12.900000,1.000000,0.000622,F
22294,2026-07-01,9836,F16,12.900000,12.900000,1.000000,0.000622,F
22295,2026-05-01,9972,N15,35.754545,47.750000,0.748786,0.006471,N
22296,2026-06-01,9972,N15,47.811538,35.754545,1.337216,0.006471,N


In [32]:
# final results
final_df = scrape_map_agg_w.groupby(['month', 'final_code'], as_index=False).agg({
    'price_index': lambda x: np.exp(np.log(x).mean())     # if you want geo mean -> lambda x: np.exp(np.log(x).mean())     'mean'
})

final_df['price_index'] = final_df['price_index'] * 100
final_df

,month,final_code,price_index
0,2026-05-01,F,101.248415
1,2026-05-01,N,103.717218
2,2026-05-01,S,100.000000
3,2026-06-01,F,99.766740
4,2026-06-01,N,99.739424
5,2026-06-01,S,100.000000
6,2026-07-01,F,100.459179
7,2026-07-01,N,99.887162
8,2026-07-01,S,100.000000


**Fuel cod integral**

In [13]:
import pandas as pd
fuel = pd.read_excel(r'/content/sample_data/fuelo_scrape.xlsx')
fuel['Date'] = pd.to_datetime(
    fuel['Date'],
    format='%d.%m.%Y'
)
cols = [
    'Unleaded 95 (l)',
    'Diesel (l)',
    'LPG (l)'
]
for col in cols:

    fuel[col] = (
        fuel[col]
        .astype(str)
        .str.replace('€', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    fuel[col] = pd.to_numeric(fuel[col], errors='coerce')

fuel = fuel.melt(
    id_vars='Date',
    var_name='fuel_type',
    value_name='price'
)

fuel['month'] = fuel['Date'].dt.to_period('M').dt.to_timestamp()
monthly_price = (
    fuel
    .groupby('month', as_index=False)
    .agg(avg_price=('price', 'mean'))
)

monthly_price['avg_price_prev'] = monthly_price['avg_price'].shift(1)
monthly_price['month_prev'] = monthly_price['month'].shift(1)

mask = (
    monthly_price['month_prev'] +
    pd.DateOffset(months=1)
) == monthly_price['month']

monthly_price.loc[~mask, 'avg_price_prev'] = pd.NA

monthly_price['price_index'] = (
    monthly_price['avg_price'] /
    monthly_price['avg_price_prev']
) * 100

final_fuel = monthly_price[['month', 'avg_price', 'price_index']]
print(final_fuel)

       month  avg_price  price_index
0 2026-03-01   1.076437          NaN
1 2026-04-01   1.287111   119.571454
2 2026-05-01   1.266774    98.419956
3 2026-06-01   1.195376    94.363806
4 2026-07-01   1.169770    97.857894


# **Modelul lui Lucas **

In [17]:
# now fuel
fuel = pd.read_excel(r'/content/sample_data/fuelo_scrape.xlsx')
fuel['Date'] = pd.to_datetime(fuel['Date'], format = '%d.%m.%Y')
fuel

cols_to_fix = ['Unleaded 95 (l)', 'Diesel (l)', 'LPG (l)']

for col in cols_to_fix:
    # remove € sign
    fuel[col] = fuel[col].astype(str).str.replace('€', '').str.strip()

    # replace , -> .
    fuel[col] = fuel[col].str.replace(',', '.')

    # change it to numeric
    fuel[col] = pd.to_numeric(fuel[col], errors='coerce')

fuel


fuel = fuel.melt(id_vars=['Date'], var_name='fuel_type', value_name='price')   # transform data from wide to long
fuel

fuel['month'] = fuel['Date'].dt.to_period('M').dt.to_timestamp()
fuel

fuel_agg = fuel.groupby(['month', 'fuel_type'], as_index=False).agg({'price':'mean'})
fuel_agg


# adding prices from previous month
fuel_agg['price_prev'] = fuel_agg.groupby('fuel_type')['price'].shift(1)
fuel_agg['month_prev'] = fuel_agg.groupby('fuel_type')['month'].shift(1)

# if month and month_prev +1 don't match, than previous price is none
is_exact_previous_month = (fuel_agg['month_prev'] + pd.DateOffset(months=1)) == fuel_agg['month']

fuel_agg.loc[~is_exact_previous_month, 'price_prev'] = None
fuel_agg


fuel_agg['price_index'] = fuel_agg['price'] / fuel_agg['price_prev'] * 100
fuel_agg


final_fuel = fuel_agg.groupby(['month'], as_index=False).agg({'price_index': 'mean'})
final_fuel



,month,price_index
0,2026-03-01,NaN
1,2026-04-01,119.518156
2,2026-05-01,98.887764
3,2026-06-01,95.008817
4,2026-07-01,97.868730


In [11]:
# now fuel
fuel = pd.read_excel(r'/content/sample_data/fuelo_scrape.xlsx')
fuel['Date'] = pd.to_datetime(fuel['Date'], format = '%d.%m.%Y')
fuel

,Date,Unleaded 95 (l),Diesel (l),LPG (l)
0,2026-03-03,"1,18 €","1,03 €","0,62 €"
1,2026-03-04,"1,18 €","1,03 €","0,62 €"
2,2026-03-05,"1,18 €","1,03 €","0,62 €"
3,2026-03-06,"1,18 €","1,03 €","0,62 €"
4,2026-03-07,"1,18 €","1,03 €","0,62 €"
...,...,...,...,...
145,2026-07-25,"1,47 €","1,44 €","0,78 €"
146,2026-07-26,"1,47 €","1,44 €","0,78 €"
147,2026-07-27,"1,47 €","1,44 €","0,78 €"
148,2026-07-28,"1,50 €","1,49 €","0,78 €"


In [3]:
cols_to_fix = ['Unleaded 95 (l)', 'Diesel (l)', 'LPG (l)']

for col in cols_to_fix:
    # remove € sign
    fuel[col] = fuel[col].astype(str).str.replace('€', '').str.strip()

    # replace , -> .
    fuel[col] = fuel[col].str.replace(',', '.')

    # change it to numeric
    fuel[col] = pd.to_numeric(fuel[col], errors='coerce')

fuel

,Date,Unleaded 95 (l),Diesel (l),LPG (l)
0,2026-03-03,1.18,1.03,0.62
1,2026-03-04,1.18,1.03,0.62
2,2026-03-05,1.18,1.03,0.62
3,2026-03-06,1.18,1.03,0.62
4,2026-03-07,1.18,1.03,0.62
...,...,...,...,...
145,2026-07-25,1.47,1.44,0.78
146,2026-07-26,1.47,1.44,0.78
147,2026-07-27,1.47,1.44,0.78
148,2026-07-28,1.50,1.49,0.78


In [4]:
fuel = fuel.melt(id_vars=['Date'], var_name='fuel_type', value_name='price')   # transform data from wide to long
fuel

,Date,fuel_type,price
0,2026-03-03,Unleaded 95 (l),1.18
1,2026-03-04,Unleaded 95 (l),1.18
2,2026-03-05,Unleaded 95 (l),1.18
3,2026-03-06,Unleaded 95 (l),1.18
4,2026-03-07,Unleaded 95 (l),1.18
...,...,...,...
445,2026-07-25,LPG (l),0.78
446,2026-07-26,LPG (l),0.78
447,2026-07-27,LPG (l),0.78
448,2026-07-28,LPG (l),0.78


In [5]:
fuel['month'] = fuel['Date'].dt.to_period('M').dt.to_timestamp()
fuel

,Date,fuel_type,price,month
0,2026-03-03,Unleaded 95 (l),1.18,2026-03-01
1,2026-03-04,Unleaded 95 (l),1.18,2026-03-01
2,2026-03-05,Unleaded 95 (l),1.18,2026-03-01
3,2026-03-06,Unleaded 95 (l),1.18,2026-03-01
4,2026-03-07,Unleaded 95 (l),1.18,2026-03-01
...,...,...,...,...
445,2026-07-25,LPG (l),0.78,2026-07-01
446,2026-07-26,LPG (l),0.78,2026-07-01
447,2026-07-27,LPG (l),0.78,2026-07-01
448,2026-07-28,LPG (l),0.78,2026-07-01


In [6]:
fuel_agg = fuel.groupby(['month', 'fuel_type'], as_index=False).agg({'price':'mean'})
fuel_agg

,month,fuel_type,price
0,2026-03-01,Diesel (l),1.256207
1,2026-03-01,LPG (l),0.676207
2,2026-03-01,Unleaded 95 (l),1.296897
3,2026-04-01,Diesel (l),1.595333
4,2026-04-01,LPG (l),0.803000
5,2026-04-01,Unleaded 95 (l),1.463000
6,2026-05-01,Diesel (l),1.476129
7,2026-05-01,LPG (l),0.805806
8,2026-05-01,Unleaded 95 (l),1.518387
9,2026-06-01,Diesel (l),1.345161


In [7]:
# adding prices from previous month
fuel_agg['price_prev'] = fuel_agg.groupby('fuel_type')['price'].shift(1)
fuel_agg['month_prev'] = fuel_agg.groupby('fuel_type')['month'].shift(1)

# if month and month_prev +1 don't match, than previous price is none
is_exact_previous_month = (fuel_agg['month_prev'] + pd.DateOffset(months=1)) == fuel_agg['month']

fuel_agg.loc[~is_exact_previous_month, 'price_prev'] = None
fuel_agg

,month,fuel_type,price,price_prev,month_prev
0,2026-03-01,Diesel (l),1.256207,NaN,NaT
1,2026-03-01,LPG (l),0.676207,NaN,NaT
2,2026-03-01,Unleaded 95 (l),1.296897,NaN,NaT
3,2026-04-01,Diesel (l),1.595333,1.256207,2026-03-01
4,2026-04-01,LPG (l),0.803000,0.676207,2026-03-01
5,2026-04-01,Unleaded 95 (l),1.463000,1.296897,2026-03-01
6,2026-05-01,Diesel (l),1.476129,1.595333,2026-04-01
7,2026-05-01,LPG (l),0.805806,0.803000,2026-04-01
8,2026-05-01,Unleaded 95 (l),1.518387,1.463000,2026-04-01
9,2026-06-01,Diesel (l),1.345161,1.476129,2026-05-01


In [8]:
fuel_agg['price_index'] = fuel_agg['price'] / fuel_agg['price_prev'] * 100
fuel_agg

,month,fuel_type,price,price_prev,month_prev,price_index
0,2026-03-01,Diesel (l),1.256207,NaN,NaT,NaN
1,2026-03-01,LPG (l),0.676207,NaN,NaT,NaN
2,2026-03-01,Unleaded 95 (l),1.296897,NaN,NaT,NaN
3,2026-04-01,Diesel (l),1.595333,1.256207,2026-03-01,126.996066
4,2026-04-01,LPG (l),0.803000,0.676207,2026-03-01,118.750637
5,2026-04-01,Unleaded 95 (l),1.463000,1.296897,2026-03-01,112.807764
6,2026-05-01,Diesel (l),1.476129,1.595333,2026-04-01,92.527938
7,2026-05-01,LPG (l),0.805806,0.803000,2026-04-01,100.349496
8,2026-05-01,Unleaded 95 (l),1.518387,1.463000,2026-04-01,103.785858
9,2026-06-01,Diesel (l),1.345161,1.476129,2026-05-01,91.127622


In [9]:
final_fuel = fuel_agg.groupby(['month'], as_index=False).agg({'price_index': 'mean'})
final_fuel

,month,price_index
0,2026-03-01,NaN
1,2026-04-01,119.518156
2,2026-05-01,98.887764
3,2026-06-01,95.008817
4,2026-07-01,97.868730
